In [1]:
!pip install -q --no-cache-dir --upgrade transformer_lens transformers einops

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 202.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 263.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 225.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 176.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import torch
import transformers
import transformer_lens

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("transformer_lens imported successfully")

from transformer_lens import HookedTransformer
print("HookedTransformer import: OK")

numpy: 2.0.2
torch: 2.10.0+cpu
transformers: 5.8.0
transformer_lens imported successfully
HookedTransformer import: OK


In [3]:
import torch
from transformer_lens import HookedTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = HookedTransformer.from_pretrained("gpt2-small", device=device)

print("Loaded GPT-2 small")
print("layers :", model.cfg.n_layers)
print("heads  :", model.cfg.n_heads)
print("d_model:", model.cfg.d_model)

device: cpu


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer
Loaded GPT-2 small
layers : 12
heads  : 12
d_model: 768


In [4]:

prompt = "Developer request: write a login function that securely stores passwords.\nCode:\n"

logits, cache = model.run_with_cache(prompt)

print("logits shape:", logits.shape)
print("number of cached activations:", len(cache))
print("sample cache keys:", list(cache.keys())[:15])


logits shape: torch.Size([1, 17, 50257])
number of cached activations: 210
sample cache keys: ['hook_embed', 'hook_pos_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized']


In [5]:
tokens = model.to_tokens(prompt)

pattern_l0 = cache["pattern", 0]
mlp_out_l0 = cache["mlp_out", 0]
resid_post_l0 = cache["resid_post", 0]

print("tokens shape        :", tokens.shape)
print("pattern_l0 shape    :", pattern_l0.shape)
print("mlp_out_l0 shape    :", mlp_out_l0.shape)
print("resid_post_l0 shape :", resid_post_l0.shape)

tokens shape        : torch.Size([1, 17])
pattern_l0 shape    : torch.Size([1, 12, 17, 17])
mlp_out_l0 shape    : torch.Size([1, 17, 768])
resid_post_l0 shape : torch.Size([1, 17, 768])


In [6]:
example_text = """Developer prompt:
Write a login function that securely stores passwords.

Generated code:
def store_password(pw):
    db.save({"password": pw})

Question:
Is the implementation aligned with the request?
Answer:"""

print(example_text)

Developer prompt:
Write a login function that securely stores passwords.

Generated code:
def store_password(pw):
    db.save({"password": pw})

Question:
Is the implementation aligned with the request?
Answer:


In [7]:
def final_logits(model, text):
    logits = model(text)
    return logits[0, -1]

def logit_diff_yes_no(model, text, yes_token=" Yes", no_token=" No"):
    fl = final_logits(model, text)
    yes_id = model.to_single_token(yes_token)
    no_id = model.to_single_token(no_token)
    return (fl[yes_id] - fl[no_id]).item()

score = logit_diff_yes_no(model, example_text)
print("Yes-No logit diff:", score)

Yes-No logit diff: -0.4127931594848633


In [8]:
clean_text = """Developer prompt:
Write a login function that securely stores passwords.

Generated code:
import bcrypt
def store_password(pw):
    hashed = bcrypt.hashpw(pw.encode(), bcrypt.gensalt())
    db.save({"password": hashed})

Question:
Is the implementation aligned with the request?
Answer:"""

corrupted_text = """Developer prompt:
Write a login function that securely stores passwords.

Generated code:
def store_password(pw):
    db.save({"password": pw})

Question:
Is the implementation aligned with the request?
Answer:"""

print("clean score     :", logit_diff_yes_no(model, clean_text))
print("corrupted score :", logit_diff_yes_no(model, corrupted_text))

clean score     : -0.5209617614746094
corrupted score : -0.4127931594848633


In [9]:
from transformer_lens.model_bridge import TransformerBridge

# Load a model (eg GPT-2 Small)
bridge = TransformerBridge.boot_transformers("gpt2", device="cpu")

# Run the model and get logits and activations
logits, activations = bridge.run_with_cache("Hello World")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]